# Miniaturas profesionales desde MMPK

Notebook independiente del APRX. Toma cada `.mmpk` desde `./mmpk`, extrae la miniatura interna (`esriinfo/thumbnail/thumbnail.*`) y genera una miniatura final con estilo consistente para ArcGIS Online / Portal / Field Maps.

Criterios aplicados desde la guia oficial de Esri sobre estilo y marca de thumbnails: menos texto, contexto visual, pista de contenido, branding organizacional y consistencia entre items.

## 1. Configuracion

In [ ]:
from datetime import datetime
from pathlib import Path
import re
import zipfile

from PIL import Image, ImageChops, ImageDraw, ImageFilter, ImageFont

ROOT_DIR = Path.cwd()
MMPK_DIR = ROOT_DIR / "mmpk"
OUTPUT_DIR = MMPK_DIR / "thumbnails"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ajusta estos valores para mantener una linea grafica consistente.
BRAND_NAME = "PAO OFFLINE"
CONTENT_LABEL = "Mobile Map Package"
APP_LABEL = "Field Maps"

# 3:2 funciona bien en cards de Portal/ArcGIS Online y conserva legibilidad en mobile.
CANVAS_SIZE = (1200, 800)

COLORS = {
    "ink": (8, 36, 48),
    "accent": (35, 130, 115),
    "paper": (246, 248, 249),
    "panel": (255, 255, 255),
    "line": (205, 214, 218),
    "soft_text": (220, 242, 238),
    "text": (255, 255, 255),
}

mmpk_files = sorted(MMPK_DIR.glob("*.mmpk"))
if not mmpk_files:
    raise FileNotFoundError(f"No se encontraron MMPK en: {MMPK_DIR}")

mmpk_files

## 2. Helpers

In [ ]:
def sanitize_filename(value):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", value).strip("_")


def title_from_mmpk(mmpk_path):
    return mmpk_path.stem.replace("_", " ").strip()


def find_mmpk_thumbnail(zip_file):
    entries = [entry for entry in zip_file.infolist() if not entry.is_dir()]
    entries_by_name = {entry.filename.replace("\\", "/").lower(): entry for entry in entries}

    for preferred_path in (
        "esriinfo/thumbnail/thumbnail.png",
        "esriinfo/thumbnail/thumbnail.jpg",
        "esriinfo/thumbnail/thumbnail.jpeg",
    ):
        entry = entries_by_name.get(preferred_path)
        if entry:
            return entry

    image_extensions = (".png", ".jpg", ".jpeg")
    candidates = [
        entry
        for entry in entries
        if "thumb" in entry.filename.lower() and entry.filename.lower().endswith(image_extensions)
    ]
    return max(candidates, key=lambda entry: entry.file_size) if candidates else None


def extract_mmpk_thumbnail(mmpk_path, output_dir):
    with zipfile.ZipFile(mmpk_path, "r") as package:
        thumbnail_entry = find_mmpk_thumbnail(package)
        if thumbnail_entry is None:
            raise FileNotFoundError(f"No se encontro miniatura dentro de: {mmpk_path}")

        suffix = Path(thumbnail_entry.filename).suffix.lower() or ".png"
        output_path = output_dir / f"{sanitize_filename(mmpk_path.stem)}_source{suffix}"
        with package.open(thumbnail_entry, "r") as source, open(output_path, "wb") as target:
            target.write(source.read())

    return output_path, thumbnail_entry.filename


def load_font(size, bold=False):
    candidates = [
        r"C:\Windows\Fonts\arialbd.ttf" if bold else r"C:\Windows\Fonts\arial.ttf",
        r"C:\Windows\Fonts\segoeuib.ttf" if bold else r"C:\Windows\Fonts\segoeui.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size=size)
        except OSError:
            continue
    return ImageFont.load_default()


def wrap_text(draw, text, font, max_width, max_lines=2):
    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if draw.textbbox((0, 0), candidate, font=font)[2] <= max_width:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word

    if current:
        lines.append(current)

    return lines[:max_lines]


def crop_white_margin(image, tolerance=12, padding=18):
    rgb = image.convert("RGB")
    background = Image.new("RGB", rgb.size, rgb.getpixel((0, 0)))
    diff = ImageChops.difference(rgb, background).convert("L")
    diff = diff.point(lambda value: 255 if value > tolerance else 0)
    bbox = diff.getbbox()

    if not bbox:
        return rgb

    left, top, right, bottom = bbox
    left = max(0, left - padding)
    top = max(0, top - padding)
    right = min(rgb.width, right + padding)
    bottom = min(rgb.height, bottom + padding)
    return rgb.crop((left, top, right, bottom))

## 3. Plantilla de miniatura

In [ ]:
def make_professional_thumbnail(source_image_path, output_path, title, updated_at):
    width, height = CANVAS_SIZE
    canvas = Image.new("RGB", CANVAS_SIZE, COLORS["paper"])
    draw = ImageDraw.Draw(canvas)

    top_h = 176
    bottom_h = 132
    side_accent_w = 20
    margin = 54

    draw.rectangle((0, 0, width, top_h), fill=COLORS["ink"])
    draw.rectangle((0, height - bottom_h, width, height), fill=COLORS["ink"])
    draw.rectangle((0, top_h, side_accent_w, height - bottom_h), fill=COLORS["accent"])

    source = crop_white_margin(Image.open(source_image_path))

    panel = (margin, top_h + 18, width - margin, height - bottom_h - 18)
    panel_w = panel[2] - panel[0]
    panel_h = panel[3] - panel[1]

    shadow = Image.new("RGBA", CANVAS_SIZE, (0, 0, 0, 0))
    shadow_draw = ImageDraw.Draw(shadow)
    shadow_draw.rectangle((panel[0] + 8, panel[1] + 10, panel[2] + 8, panel[3] + 10), fill=(0, 0, 0, 34))
    shadow = shadow.filter(ImageFilter.GaussianBlur(8))
    canvas = Image.alpha_composite(canvas.convert("RGBA"), shadow).convert("RGB")
    draw = ImageDraw.Draw(canvas)

    draw.rectangle(panel, fill=COLORS["panel"], outline=COLORS["line"], width=2)

    image_margin = 28
    image_box = (
        panel[0] + image_margin,
        panel[1] + image_margin,
        panel[2] - image_margin,
        panel[3] - image_margin,
    )
    image_w = image_box[2] - image_box[0]
    image_h = image_box[3] - image_box[1]
    scale = min(image_w / source.width, image_h / source.height)
    resized = source.resize((max(1, int(source.width * scale)), max(1, int(source.height * scale))), Image.LANCZOS)
    image_x = image_box[0] + (image_w - resized.width) // 2
    image_y = image_box[1] + (image_h - resized.height) // 2
    canvas.paste(resized, (image_x, image_y))

    brand_font = load_font(34, bold=True)
    title_font = load_font(50, bold=True)
    label_font = load_font(32, bold=True)
    meta_font = load_font(26)

    draw = ImageDraw.Draw(canvas)
    draw.text((margin, 28), BRAND_NAME, fill=COLORS["text"], font=brand_font)
    y = 78
    for line in wrap_text(draw, title, title_font, width - (margin * 2), max_lines=2):
        draw.text((margin, y), line, fill=COLORS["text"], font=title_font)
        y += 58

    draw.text((margin, height - 98), f"{CONTENT_LABEL} - {APP_LABEL}", fill=COLORS["text"], font=label_font)
    draw.text((margin, height - 52), f"Actualizado: {updated_at}", fill=COLORS["soft_text"], font=meta_font)

    canvas.save(output_path, format="PNG", optimize=True)
    return output_path

## 4. Generar miniaturas

In [ ]:
thumbnail_rows = []

for mmpk_path in mmpk_files:
    source_thumbnail, internal_path = extract_mmpk_thumbnail(mmpk_path, OUTPUT_DIR)
    final_thumbnail = OUTPUT_DIR / f"{sanitize_filename(mmpk_path.stem)}_thumbnail.png"
    updated_at = datetime.fromtimestamp(mmpk_path.stat().st_mtime).astimezone().strftime("%d-%m-%Y %H:%M")

    make_professional_thumbnail(
        source_image_path=source_thumbnail,
        output_path=final_thumbnail,
        title=title_from_mmpk(mmpk_path),
        updated_at=updated_at,
    )

    thumbnail_rows.append(
        {
            "mmpk": mmpk_path.name,
            "internal_thumbnail": internal_path,
            "source_thumbnail": str(source_thumbnail),
            "final_thumbnail": str(final_thumbnail),
        }
    )

thumbnail_rows

## 5. Vista previa

In [ ]:
from IPython.display import Image as NotebookImage, display

for row in thumbnail_rows:
    print(row["mmpk"])
    display(NotebookImage(filename=row["final_thumbnail"]))

## 6. Actualizar item en Portal opcional

In [ ]:
# Ejecuta esta celda solo si quieres actualizar la miniatura de un item existente.
# Completa el ID del item por cada nombre de MMPK.

UPDATE_PORTAL_ITEM = False
ITEM_IDS_BY_MMPK = {
    "MLP_SIG_PAO_LAYOUT_OFFLINE_v4.mmpk": "108e95e1d79d4935a93bbea3db536d59",
}

if UPDATE_PORTAL_ITEM:
    import json
    from arcgis.gis import GIS

    cred = json.load(open("./Json/AMSA.json", encoding="utf-8"))["PAO"]
    gis = GIS(cred["url"], cred["user"], cred["pass"])

    for row in thumbnail_rows:
        item_id = ITEM_IDS_BY_MMPK.get(row["mmpk"])
        if not item_id:
            print(f"Sin item id configurado para: {row['mmpk']}")
            continue

        item = gis.content.get(item_id)
        if item is None:
            raise ValueError(f"No se encontro item: {item_id}")

        updated = item.update(thumbnail=row["final_thumbnail"])
        print(f"{row['mmpk']}: thumbnail actualizado = {updated}")